In [ ]:
import yaml
import json
import os

from src.datasets.dataset_from_tg import data_full_routine
from src.transformers.transformers_utils import init_pretrained_model, tokenize_function

In [2]:
CONFIG_DIR = "configs/tune"
CONFIG_NAME = "saiga_gemma2_10b.default.yaml" # parse arg
CONFIG_PATH = os.path.join(CONFIG_DIR, CONFIG_NAME)

In [3]:
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)
    
random_seed = config["RANDOM_SEED"]
model_name = config["MODEL_NAME"]

data_path = os.path.join(config["DATA_DIR"], config["DATA_NAME"])
with open(data_path, "r") as f:
    raw_data = json.load(f)

name = config["RESPONSE_NAME"]

In [ ]:
model, tokenizer = init_pretrained_model(model_name, random_seed, device_map="cuda:0")

dataset = data_full_routine(raw_data, name)
tokenized_dataset = dataset.map(lambda x: tokenize_function(sample=x, tokenizer=tokenizer), batched=True)

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

In [ ]:
train_test_split = dataset.train_test_split(test_size=0.3, shuffle=True,)

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./temp",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_test_split["train"],
    eval_dataset=train_test_split["test"],
)

trainer.train()